# Phishing Detector Starter Notebook

Catalog files, build labels, parse emails, de‑duplicate, and run **Leave‑One‑Dataset‑Out (LODO)** baselines.
> Expectation: You place your CSV files under `data/raw/`. Update `label_map.csv` as needed.

## Environment & Paths

In [ ]:

import os, re, json, math, glob, hashlib, email, html, string, itertools, random
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional

import pandas as pd
import numpy as np

try:
    from bs4 import BeautifulSoup
    _HAS_BS4 = True
except Exception:
    _HAS_BS4 = False

# Project paths (relative to this notebook)
DATA_RAW = Path("./data/raw")
OUT_DIR = Path(".")
ART_DIR = OUT_DIR / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)

LABEL_MAP_PATH = OUT_DIR / "label_map.csv"
if LABEL_MAP_PATH.exists():
    label_map_df = pd.read_csv(LABEL_MAP_PATH)
    display(label_map_df.head())
else:
    display("label_map.csv not found — create it or copy from your project root.")


## Utilities: URL extraction, HTML → text, RFC822 parsing, normalization

In [ ]:

URL_RE = re.compile(r"\b((?:https?://|www\.)[^\s<>\"']+)\b", re.IGNORECASE)

def extract_urls(text: str) -> List[str]:
    if not isinstance(text, str) or not text:
        return []
    return [u.rstrip(').,;:') for u in URL_RE.findall(text)]

def html_to_text(html_str: str) -> str:
    if not isinstance(html_str, str) or not html_str:
        return ""
    if _HAS_BS4:
        try:
            soup = BeautifulSoup(html_str, "html.parser")
            for tag in soup(["script", "style", "noscript"]):
                tag.decompose()
            return soup.get_text(" ", strip=True)
        except Exception:
            pass
    s = re.sub(r"<(script|style)[^>]*>.*?</\1>", " ", html_str, flags=re.I|re.S)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def parse_email(raw: str) -> Tuple[str, str, str]:
    if not isinstance(raw, str):
        raw = str(raw)
    try:
        msg = email.message_from_string(raw)
        subj = msg.get('Subject') or ""
        body_text, body_html = "", ""
        if msg.is_multipart():
            for part in msg.walk():
                ctype = (part.get_content_type() or "").lower()
                disp = (part.get('Content-Disposition') or "").lower()
                if "attachment" in disp:
                    continue
                try:
                    payload = part.get_payload(decode=True)
                    if payload is None:
                        continue
                    charset = part.get_content_charset() or "utf-8"
                    text = payload.decode(charset, errors="replace")
                except Exception:
                    continue
                if ctype == "text/plain":
                    body_text += "\n" + text
                elif ctype == "text/html":
                    body_html += "\n" + text
        else:
            payload = msg.get_payload(decode=True)
            if payload is None:
                text = msg.get_payload()
            else:
                charset = msg.get_content_charset() or "utf-8"
                text = payload.decode(charset, errors="replace")
            if isinstance(text, str) and ("<html" in text.lower() or "</p>" in text.lower()):
                body_html = text or ""
            else:
                body_text = text or ""
        if body_html and not body_text:
            body_text = html_to_text(body_html)
        return (subj or "").strip(), (body_text or "").strip(), html_to_text(body_html)
    except Exception:
        return "", raw, ""

def normalize_text(t: str) -> str:
    t = (t or "").lower()
    t = html.unescape(t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

def text_signature(subject: str, body: str) -> str:
    norm = normalize_text(subject + " " + body)
    return hashlib.md5(norm.encode("utf-8")).hexdigest()

def jaccard_similarity(a: set, b: set) -> float:
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union else 0.0

def char_shingles(text: str, k: int = 5) -> set:
    text = normalize_text(text)
    if len(text) < k: return {text}
    return {text[i:i+k] for i in range(len(text)-k+1)}


## Load & Parse CSVs

Walk `data/raw/`, infer the likely text column, parse email, extract URLs, and map labels from `label_map.csv`.

In [ ]:

TEXT_CANDIDATE_COLS = [
    "text","Text","message","Message","raw","Raw","email","Email","content","Content",
    "body","Body","mail","Mail","data","Data"
]
SUBJECT_CANDIDATE_COLS = ["subject","Subject","SUBJECT","subj","Subj"]

def pick_first_existing(candidates: List[str], cols: List[str]) -> Optional[str]:
    for c in candidates:
        if c in cols:
            return c
    return None

def dataset_key_from_fname(path: Path) -> str:
    base = path.name
    base = re.sub(r"\.csv(\.gz)?$", "", base, flags=re.I)
    return base

def load_all_raw(data_dir: Path, label_map_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    files = sorted(data_dir.glob("*.csv")) + sorted(data_dir.glob("*.csv.gz"))
    if not files:
        print("No CSVs found in", data_dir)
    for f in files:
        dk = dataset_key_from_fname(f)
        lm = label_map_df[label_map_df["dataset_key"] == dk] if not label_map_df.empty else pd.DataFrame()
        if lm.empty:
            print(f"[skip] {f.name}: dataset_key not in label_map.csv")
            continue
        mapped = lm.iloc[0]["mapped_binary_label"]
        if mapped not in ("phish","not_phish"):
            print(f"[skip] {f.name}: mapped label is {mapped}")
            continue
        try:
            df = pd.read_csv(f, encoding="utf-8", dtype=str, keep_default_na=False, na_values=[])
        except Exception:
            df = pd.read_csv(f, encoding="latin-1", dtype=str, keep_default_na=False, na_values=[])
        text_col = pick_first_existing(TEXT_CANDIDATE_COLS, list(df.columns))
        subj_col = pick_first_existing(SUBJECT_CANDIDATE_COLS, list(df.columns))
        if text_col is None:
            text_col = max(df.columns, key=lambda c: df[c].astype(str).str.len().mean())
        for _, r in df.iterrows():
            raw = str(r.get(text_col, ""))
            subj_raw = str(r.get(subj_col, "")) if subj_col else ""
            subj, body_txt, body_html_txt = parse_email(raw if len(raw) > len(subj_raw) else subj_raw + "\n" + raw)
            subj = subj or subj_raw or ""
            urls = extract_urls(subj + " " + body_txt)
            rows.append({
                "dataset_key": dk,
                "file_name": f.name,
                "subject": subj,
                "body_text": body_txt if body_txt else raw,
                "label": mapped,
                "urls": urls
            })
    return pd.DataFrame(rows)

all_df = load_all_raw(DATA_RAW, label_map_df if 'label_map_df' in globals() else pd.DataFrame())
print("Loaded rows:", len(all_df))
display(all_df.head())


## De-duplication

Exact duplicates removal; optional near‑dup via Jaccard over char 5‑grams (off by default).

In [ ]:

def dedup(df: pd.DataFrame, near_dup: bool=False, jaccard_threshold: float=0.95) -> pd.DataFrame:
    if df.empty:
        return df
    sigs = (df["subject"].astype(str) + " " + df["body_text"].astype(str)).map(text_signature)
    df = df.assign(_sig=sigs)
    df = df.drop_duplicates(subset=["_sig"]).drop(columns=["_sig"])
    if not near_dup:
        return df
    kept = []
    for dk, g in df.groupby("dataset_key"):
        g = g.copy()
        shingles_list = [char_shingles((r.subject or "") + " " + (r.body_text or "")) for r in g.itertuples(index=False)]
        keep_mask = np.ones(len(g), dtype=bool)
        for i in range(len(g)):
            if not keep_mask[i]:
                continue
            for j in range(i+1, len(g)):
                if not keep_mask[j]:
                    continue
                sim = jaccard_similarity(shingles_list[i], shingles_list[j])
                if sim >= jaccard_threshold:
                    keep_mask[j] = False
        kept.append(g[keep_mask])
    return pd.concat(kept, ignore_index=True)

dedup_df = dedup(all_df, near_dup=False)
print("After exact-dup removal:", len(dedup_df))
display(dedup_df.head())


## Leave‑One‑Dataset‑Out (LODO) splits

Hold out one dataset for testing; stratified train/val on the rest.

In [ ]:

from sklearn.model_selection import train_test_split

def lodo_splits(df: pd.DataFrame, test_dataset: str, val_size: float=0.1, seed: int=42):
    df = df.copy()
    test_df = df[df["dataset_key"] == test_dataset]
    rest_df = df[df["dataset_key"] != test_dataset]
    if rest_df.empty or test_df.empty:
        raise ValueError("Not enough data for LODO with test_dataset=" + test_dataset)
    X = rest_df.index.values
    y = (rest_df["label"] == "phish").astype(int).values
    train_idx, val_idx = train_test_split(X, test_size=val_size, random_state=seed, stratify=y)
    train_df = rest_df.loc[train_idx]
    val_df = rest_df.loc[val_idx]
    return train_df, val_df, test_df


## Baseline model: char TF‑IDF + Logistic Regression

Train per LODO fold; save models and a results CSV.

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

def join_text(df: pd.DataFrame):
    return (df["subject"].fillna("") + " \n " + df["body_text"].fillna("")).tolist()

def train_eval_baseline(train_df, val_df, test_df, fold_name="fold"):
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(3,5), max_features=200000)),
        ("lr", LogisticRegression(max_iter=400, class_weight="balanced"))
    ])
    y_tr = (train_df["label"]=="phish").astype(int).values
    y_va = (val_df["label"]=="phish").astype(int).values
    y_te = (test_df["label"]=="phish").astype(int).values

    pipe.fit(join_text(train_df), y_tr)
    val_probs = pipe.predict_proba(join_text(val_df))[:,1]
    test_probs = pipe.predict_proba(join_text(test_df))[:,1]

    metrics = {
        "val_roc_auc": float(roc_auc_score(y_va, val_probs)) if len(np.unique(y_va))>1 else float("nan"),
        "val_pr_auc": float(average_precision_score(y_va, val_probs)) if len(np.unique(y_va))>1 else float("nan"),
        "test_roc_auc": float(roc_auc_score(y_te, test_probs)) if len(np.unique(y_te))>1 else float("nan"),
        "test_pr_auc": float(average_precision_score(y_te, test_probs)) if len(np.unique(y_te))>1 else float("nan"),
    }
    print(f"[{fold_name}] Validation ROC-AUC: {metrics['val_roc_auc']:.4f}, PR-AUC: {metrics['val_pr_auc']:.4f}")
    print(f"[{fold_name}] Test ROC-AUC: {metrics['test_roc_auc']:.4f}, PR-AUC: {metrics['test_pr_auc']:.4f}")
    return pipe, metrics

def run_lodo(df: pd.DataFrame, deduped: bool=True):
    work = dedup(df) if deduped else df
    datasets = sorted(work["dataset_key"].unique())
    results = []
    for holdout in datasets:
        try:
            tr, va, te = lodo_splits(work, holdout)
        except Exception as e:
            print(f"[skip fold {holdout}] {e}")
            continue
        model, metrics = train_eval_baseline(tr, va, te, fold_name=holdout)
        import joblib, os
        model_path = ART_DIR / f"baseline_tfidf_lr__holdout_{holdout}.joblib"
        joblib.dump(model, model_path)
        metrics.update({
            "holdout_dataset": holdout,
            "n_train": len(tr),
            "n_val": len(va),
            "n_test": len(te),
            "model_path": str(model_path)
        })
        results.append(metrics)
    res_df = pd.DataFrame(results)
    if not res_df.empty:
        res_csv = OUT_DIR / "lodo_results.csv"
        res_df.to_csv(res_csv, index=False)
        print("Saved LODO results to", res_csv)
    return res_df

# Example (uncomment after data is loaded into data/raw/):
# lodo_df = run_lodo(dedup_df, deduped=True)
# display(lodo_df)


## Per‑dataset error analysis (optional)

After training a specific holdout model, inspect top false negatives/positives to guide feature additions.

In [ ]:

def error_table(model, test_df: pd.DataFrame, k: int=20):
    y_true = (test_df["label"]=="phish").astype(int).values
    probs = model.predict_proba(join_text(test_df))[:,1]
    preds = (probs>=0.5).astype(int)
    test_df = test_df.copy()
    test_df["prob_phish"] = probs
    test_df["pred"] = preds
    fns = test_df[(test_df["label"]=="phish") & (test_df["pred"]==0)].sort_values("prob_phish", ascending=True).head(k)
    fps = test_df[(test_df["label"]=="not_phish") & (test_df["pred"]==1)].sort_values("prob_phish", ascending=False).head(k)
    display({"false_negatives": fns[["dataset_key","subject","prob_phish"]],
             "false_positives": fps[["dataset_key","subject","prob_phish"]]})
